## Trip extraction: leisure OD tables with activity type and language codes

This notebook builds **origin–destination (OD) trip tables** for leisure mobility and assigns each trip to the selected **NPVM 2017 traffic zones**. The output provides the demand-side input for the destination choice estimation.

The notebook produces the observed chosen destinations that are later combined with sampled non-chosen alternatives, travel impedance variables, and destination attractivity predictors in the destination choice model.

### Data sources

The notebook uses the Swiss Microcensus Mobility and Transport data for 2015 and 2021:

- **MTMC 2015 + 2021**
  - Short trips: `wegeinland.csv` + `zielpersonen.csv`
  - Day trips: `tagesreisen.csv` + `zielpersonen.csv`

The spatial reference layer is:

- **Selected NPVM 2017 traffic zones**
  - `TZ/TZ.gpkg`
  - layer: `TZ_fixed`
  - zone identifier: `npvm_id`

All spatial operations are performed in **EPSG:2056**.

### Leisure definition

Short and long leisure trips are handled differently.

#### Short trips: `WegeInland`

Short leisure trips are selected from `wegeinland.csv` using:

- `wzweck3 == 8`

which corresponds to leisure activities.

In addition, the notebook keeps only outward trips:

- `wzweck2 == 1`

This is important because the destination choice model requires the observed destination to be the actual leisure place. By keeping only outward trips, return trips from a leisure activity back home are excluded. Otherwise, a trip related to leisure could have the home location as its destination, which would not represent a chosen leisure destination.

Therefore, short leisure trips are defined as:

- leisure purpose: `wzweck3 == 8`
- outward trip / Hinweg: `wzweck2 == 1`

#### Long/day trips: `Tagesreisen`

Day trips are selected from `tagesreisen.csv`.

A day trip is kept only if its detailed purpose can be mapped to one of the leisure activity categories used in the thesis. Non-mappable, missing, or excluded purpose codes are dropped.

The detailed purpose column is selected automatically from the available variants, for example:

- `F60800_01A`
- `f60800_01A`
- `F60800_01`
- `f60800_01`

### Age segments

Trips are split into two age groups:

- `5–64`: young
- `65+`: old

These groups are used consistently with the thesis segmentation.

### Leisure activity categories

Each trip is assigned a coarse leisure activity type stored in:

- `leisure_cat`

For short trips from `WegeInland`, the category is based on:

- `f51700_weg`

For long/day trips from `Tagesreisen`, the category is based on the detailed day-trip purpose field.

The categories used are:

- `Visits`
- `Gastronomy`
- `Outdoor`
- `Culture`
- `Sport`
- `Others`
- `NK`

The `NK` category is used only for short trips when the detailed leisure subtype is missing, unavailable, unknown, or excluded. For day trips, only trips that can be mapped to a valid leisure category are retained.

### Language codes

The notebook stores the language code of the start and destination locations.

For short trips from `WegeInland`, the fields are:

- `S_SPRACHE`
- `Z_SPRACHE`

For long/day trips from `Tagesreisen`, the fields are:

- `TRS_SPRACHE`
- `TRZ_SPRACHE`

The coding is:

- `1`: German
- `2`: French
- `3`: Italian
- `4`: Romansh

The output columns are:

- `start_lang`
- `dest_lang`

### Traffic-zone assignment

Origins and destinations are assigned to traffic zones using point-in-polygon spatial joins.

The notebook reads the corrected traffic-zone layer:

- file: `TZ/TZ.gpkg`
- layer: `TZ_fixed`

The zone identifier used in the output is:

- `npvm_id`

This is used because it is aligned with the NPVM-like IDs used by the OMX travel-cost matrices.

The assignment strategy is:

1. Convert WGS84 survey coordinates to EPSG:2056.
2. Assign each point to a traffic zone using a `within` spatial join.
3. Use an `intersects` join as fallback for unmatched boundary cases.
4. Keep only trips with both an origin and a destination zone.

### Output files

The notebook saves four OD tables to the `Trips/` folder:

- `Trips/destinations_YS.csv`: Young, Short trips
- `Trips/destinations_OS.csv`: Old, Short trips
- `Trips/destinations_YL.csv`: Young, Long/day trips
- `Trips/destinations_OL.csv`: Old, Long/day trips

The segment labels are:

- `YS`: young short trips
- `OS`: old short trips
- `YL`: young long/day trips
- `OL`: old long/day trips

### Output columns

Each output table contains the following variables:

- `HHNR`: household/person identifier from the survey
- `orig_zone`: origin traffic zone ID
- `dest_zone`: destination traffic zone ID
- `survey`: MTMC wave, either `2015` or `2021`
- `agecat`: age category, either `5-64` or `65+`
- `leisure_cat`: coarse leisure activity type
- `start_lang`: language code at the origin location
- `dest_lang`: language code at the destination location
- `dist_km`: trip distance in kilometres
- `dur_min`: trip duration in minutes
- `WP`: person weight, when available

The combination of `HHNR` and `survey` can be used to identify persons within a survey wave, for example for out-of-sample splits.

### Notes

- Short leisure trips keep only outward trips with `wzweck2 == 1`, so the destination is the leisure activity location.
- Return trips from leisure activities are excluded because their destination is often home rather than the leisure place.
- Day trips are kept only when their detailed purpose can be mapped to a valid leisure activity category.
- All point-to-zone assignments are performed in EPSG:2056.
- The resulting OD tables represent the observed chosen alternatives for the destination choice estimation.

In [ ]:
# ============================================================
# Trip extraction (leisure OD tables) with activity type + language codes
# MOD: short leisure trips = ONLY outward trips (wzweck2 == 1, Hinweg)
# MOD: assign zones using TZ.npvm_id (NPVM-like ID) so it matches OMX matrices
# EXTRA: save HHNR so (HHNR + survey) identifies persons within wave for out-of-sample splits
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

# =====================================================================
# BASE PATHS
# =====================================================================
PATH15  = "MTMZ/MZMV2015_mit_Geo/4_DB_csv/CH_CSV"
PATH21  = "MTMZ/MZMV2021_mit_Geo/4_DB_csv/4_DB_csv"
TZ_PATH = "TZ/TZ.gpkg"   # must contain NPVM 2017 zones with npvm_id aligned to matrices

os.makedirs("Trips", exist_ok=True)

CRS_WGS84  = 4326
CRS_TARGET = 2056

# =====================================================================
# LEISURE CATEGORY MAPPERS
# =====================================================================
def map_wegeinland_leisure_activity(code: float) -> str:
    """
    WegeInland leisure activity codes (f51700_weg).
    Returns one of: Visits, Gastronomy, Outdoor, Culture, Sport, Others, NK
    NK = missing/excluded/unknown/manual
    """
    if not np.isfinite(code):
        return "NK"
    code = int(code)

    if code in (-99, -98, -97):
        return "NK"

    if code == 1:
        return "Visits"
    if code == 2:
        return "Gastronomy"
    if code in (4, 5, 7, 15, 17):
        return "Outdoor"
    if code == 9:
        return "Culture"
    if code == 3:
        return "Sport"
    if code in (6, 8, 10, 11, 12, 13, 14, 16, 18, 22, 90):
        return "Others"

    return "NK"


def map_tagesreisen_leisure_activity(code: float) -> str | None:
    """
    Tagesreisen detailed purpose (f60800_01 / F60800_01A depending on file).
    Returns one of: Visits, Gastronomy, Outdoor, Culture, Sport, Others
    Returns None if missing/excluded/non-mappable.
    """
    if not np.isfinite(code):
        return None
    code = int(code)

    if code in (-99, -98, -97):
        return None

    if code == 5:
        return "Visits"
    if code == 6:
        return "Gastronomy"
    if code == 7:
        return "Sport"
    if code in (8, 9, 11, 17):
        return "Outdoor"
    if code == 12:
        return "Culture"
    if code in (10, 13, 14):
        return "Others"

    return None


# =====================================================================
# LOADING FUNCTIONS
# =====================================================================
def load_wegeinland_year(path, year):
    sep = "," if year == 2015 else ";"
    wege = pd.read_csv(f"{path}/wegeinland.csv", sep=sep, encoding="latin1", low_memory=False)
    ziel = pd.read_csv(f"{path}/zielpersonen.csv", sep=sep, encoding="latin1", low_memory=False)

    df = wege.merge(ziel[["HHNR", "alter", "WP"]], on="HHNR", how="left", suffixes=("", "_ZP"))

    df = df[df["alter"].notna()].copy()
    df["agecat"] = np.select(
        [(df["alter"] >= 5) & (df["alter"] <= 64), (df["alter"] >= 65)],
        ["5-64", "65+"],
        default="out"
    )
    df = df[df["agecat"] != "out"].copy()
    df["survey"] = year
    return df


def load_tagesreisen_year(path, year):
    sep = "," if year == 2015 else ";"
    tages = pd.read_csv(f"{path}/tagesreisen.csv", sep=sep, encoding="latin1", low_memory=False)
    ziel  = pd.read_csv(f"{path}/zielpersonen.csv", sep=sep, encoding="latin1", low_memory=False)

    ziel = ziel[["HHNR", "alter"]].copy()
    ziel = ziel[ziel["alter"].notna()].copy()
    ziel["agecat"] = np.select(
        [(ziel["alter"] >= 5) & (ziel["alter"] <= 64), (ziel["alter"] >= 65)],
        ["5-64", "65+"],
        default="out"
    )
    ziel = ziel[ziel["agecat"] != "out"][["HHNR", "agecat"]]

    df = tages.merge(ziel, on="HHNR", how="left")
    df["survey"] = year
    return df


# =====================================================================
# LOAD TZ LAYER (NPVM 2017) -> EPSG:2056
# IMPORTANT: use layer="TZ_fixed" to ensure npvm_id is the corrected one
# =====================================================================
tz = gpd.read_file(TZ_PATH, layer="TZ_fixed")
if tz.crs is None:
    raise ValueError("TZ.gpkg has no CRS set. Please set it to EPSG:2056 (LV95).")
if tz.crs.to_epsg() != CRS_TARGET:
    tz = tz.to_crs(CRS_TARGET)

if "npvm_id" not in tz.columns:
    raise ValueError("TZ.gpkg (TZ_fixed layer) must contain a 'npvm_id' column aligned to matrices.")

tz_zones = tz[["npvm_id", "geometry"]].copy()


# =====================================================================
# POINT -> ZONE ASSIGNMENT (WGS84 input ONLY)
# MOD: returns TZ.npvm_id
# =====================================================================
def assign_zones_wgs84(df: pd.DataFrame, x_col: str, y_col: str,
                       tz_gdf: gpd.GeoDataFrame, out_col: str) -> pd.DataFrame:
    """
    Assign each (x,y) point in WGS84 to a TZ npvm_id.
    Strategy: within then intersects fallback.
    Returns: DataFrame with trip_id and out_col (npvm_id), one row per trip_id when matched.
    """
    sub = df[["trip_id", x_col, y_col]].copy()

    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub[y_col] = pd.to_numeric(sub[y_col], errors="coerce")
    sub = sub[sub[x_col].notna() & sub[y_col].notna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=["trip_id", out_col])

    gdf = gpd.GeoDataFrame(
        sub[["trip_id"]],
        geometry=gpd.points_from_xy(sub[x_col], sub[y_col]),
        crs=f"EPSG:{CRS_WGS84}"
    ).to_crs(CRS_TARGET)

    # 1) within join
    sj = gpd.sjoin(gdf, tz_gdf, how="left", predicate="within")
    matched = sj[sj["npvm_id"].notna()][["trip_id", "npvm_id"]].drop_duplicates("trip_id")

    # 2) fallback intersects for remaining points
    if len(matched) < len(gdf):
        remaining = gdf[~gdf["trip_id"].isin(matched["trip_id"])].copy()
        if not remaining.empty:
            sj2 = gpd.sjoin(remaining, tz_gdf, how="left", predicate="intersects")
            matched2 = sj2[sj2["npvm_id"].notna()][["trip_id", "npvm_id"]].drop_duplicates("trip_id")
            matched = pd.concat([matched, matched2], ignore_index=True)

    matched = matched.rename(columns={"npvm_id": out_col})
    return matched


# =====================================================================
# 1) LOAD DATA (2015 + 2021)
# =====================================================================
wege_all = pd.concat(
    [load_wegeinland_year(PATH15, 2015), load_wegeinland_year(PATH21, 2021)],
    ignore_index=True
)

tages_all = pd.concat(
    [load_tagesreisen_year(PATH15, 2015), load_tagesreisen_year(PATH21, 2021)],
    ignore_index=True
)

# =====================================================================
# 2) FILTER LEISURE + ADD leisure_cat + LANGS
# =====================================================================

# ---------- SHORT (WegeInland): leisure by wzweck3 == 8 AND ONLY Hinweg wzweck2 == 1 ----------
wege_all["wzweck3"] = pd.to_numeric(wege_all["wzweck3"], errors="coerce")
wege_all["wzweck2"] = pd.to_numeric(wege_all["wzweck2"], errors="coerce")

wege_leisure = wege_all[
    (wege_all["wzweck3"] == 8) &
    (wege_all["wzweck2"] == 1)   # ONLY outward trip / Hinweg
].copy()

wege_leisure = wege_leisure[wege_leisure["agecat"].isin(["5-64", "65+"])].copy()

# leisure_cat from f51700_weg
if "f51700_weg" in wege_leisure.columns:
    wege_leisure["f51700_weg"] = pd.to_numeric(wege_leisure["f51700_weg"], errors="coerce")
    wege_leisure["leisure_cat"] = wege_leisure["f51700_weg"].apply(map_wegeinland_leisure_activity)
else:
    wege_leisure["leisure_cat"] = "NK"

# language codes (WegeInland)
wege_leisure["start_lang"] = pd.to_numeric(wege_leisure.get("S_SPRACHE"), errors="coerce")
wege_leisure["dest_lang"]  = pd.to_numeric(wege_leisure.get("Z_SPRACHE"), errors="coerce")

# ---------- LONG (Tagesreisen): keep only rows mapping to leisure_cat ----------
purpose_candidates = ["F60800_01A", "f60800_01A", "F60800_01", "f60800_01"]
purpose_col = next((c for c in purpose_candidates if c in tages_all.columns), None)
if purpose_col is None:
    raise ValueError(f"No Tagesreisen purpose column found among: {purpose_candidates}")

tages_all[purpose_col] = pd.to_numeric(tages_all[purpose_col], errors="coerce")
tages_all["leisure_cat_tmp"] = tages_all[purpose_col].apply(map_tagesreisen_leisure_activity)

tages_leisure = tages_all[tages_all["leisure_cat_tmp"].notna()].copy()
tages_leisure = tages_leisure[tages_leisure["agecat"].isin(["5-64", "65+"])].copy()
tages_leisure = tages_leisure.rename(columns={"leisure_cat_tmp": "leisure_cat"})

# language codes (Tagesreisen)
tages_leisure["start_lang"] = pd.to_numeric(tages_leisure.get("TRS_SPRACHE"), errors="coerce")
tages_leisure["dest_lang"]  = pd.to_numeric(tages_leisure.get("TRZ_SPRACHE"), errors="coerce")

# =====================================================================
# 3) ASSIGN POINTS TO ZONES (ORIGIN/DESTINATION) — WGS84 coords for ALL waves
# =====================================================================

# ---------- SHORT (WegeInland) ----------
wege_short = wege_leisure.copy().reset_index(drop=True)
wege_short["trip_id"] = np.arange(len(wege_short))

# distance/time columns for WegeInland
if "w_rdist" in wege_short.columns:
    wege_short["w_rdist"] = pd.to_numeric(wege_short["w_rdist"], errors="coerce")
if "dauer1" in wege_short.columns:
    wege_short["dur_tmp"] = pd.to_numeric(wege_short["dauer1"], errors="coerce")
elif "dauer2" in wege_short.columns:
    wege_short["dur_tmp"] = pd.to_numeric(wege_short["dauer2"], errors="coerce")
else:
    wege_short["dur_tmp"] = np.nan

# Assign zones (WGS84 -> 2056) using npvm_id
orig_zones_short = assign_zones_wgs84(wege_short, "S_X", "S_Y", tz_zones, "orig_zone_id")
dest_zones_short = assign_zones_wgs84(wege_short, "Z_X", "Z_Y", tz_zones, "dest_zone_id")

wege_short = wege_short.merge(orig_zones_short, on="trip_id", how="left")
wege_short = wege_short.merge(dest_zones_short, on="trip_id", how="left")

# keep only trips with both zones (OD needed)
wege_short = wege_short.dropna(subset=["orig_zone_id", "dest_zone_id"]).copy()

# ---------- LONG (Tagesreisen) ----------
tages_long = tages_leisure.copy().reset_index(drop=True)
tages_long["trip_id"] = np.arange(len(tages_long))

for col in ["f60700", "f61600b", "f61700b"]:
    if col in tages_long.columns:
        tages_long[col] = pd.to_numeric(tages_long[col], errors="coerce")

dist_inland = tages_long["f61700b"] if "f61700b" in tages_long.columns else pd.Series(np.nan, index=tages_long.index)
dist_inland = dist_inland.where(dist_inland >= 0, np.nan)

dist_total = tages_long["f61600b"] if "f61600b" in tages_long.columns else pd.Series(np.nan, index=tages_long.index)
dist_total = dist_total.where(dist_total >= 0, np.nan)

tages_long["dist_km_tmp"] = dist_inland.fillna(dist_total)
tages_long["dur_min_tmp"] = tages_long["f60700"] * 60 if "f60700" in tages_long.columns else np.nan

# Assign zones (WGS84 -> 2056) using npvm_id
orig_zones_long = assign_zones_wgs84(tages_long, "TRS_X", "TRS_Y", tz_zones, "orig_zone_id")
dest_zones_long = assign_zones_wgs84(tages_long, "TRZ_X", "TRZ_Y", tz_zones, "dest_zone_id")

tages_long = tages_long.merge(orig_zones_long, on="trip_id", how="left")
tages_long = tages_long.merge(dest_zones_long, on="trip_id", how="left")

# require both origin and destination
tages_long = tages_long.dropna(subset=["orig_zone_id", "dest_zone_id"]).copy()

# =====================================================================
# 4) BUILD THE 4 CLEAN DATASETS (YS, OS, YL, OL)
# =====================================================================
def build_dest_df(base_df: pd.DataFrame, agecat_value: str,
                  dist_col: str, dur_col: str) -> pd.DataFrame:
    cols = [
        "HHNR",
        "orig_zone_id", "dest_zone_id",
        "survey", "agecat",
        "leisure_cat",
        "start_lang", "dest_lang",
        dist_col, dur_col
    ]
    if "WP" in base_df.columns:
        cols.append("WP")

    df = base_df[base_df["agecat"] == agecat_value].copy()
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    df = df.rename(columns={
        "orig_zone_id": "orig_zone",
        "dest_zone_id": "dest_zone",
        dist_col: "dist_km",
        dur_col: "dur_min",
    }).reset_index(drop=True)

    # types
    df["HHNR"] = pd.to_numeric(df["HHNR"], errors="coerce").astype("Int64")
    df["orig_zone"] = pd.to_numeric(df["orig_zone"], errors="coerce").astype("Int64")
    df["dest_zone"] = pd.to_numeric(df["dest_zone"], errors="coerce").astype("Int64")

    df["start_lang"] = pd.to_numeric(df.get("start_lang"), errors="coerce").astype("Int64")
    df["dest_lang"]  = pd.to_numeric(df.get("dest_lang"), errors="coerce").astype("Int64")

    if "dist_km" in df.columns:
        df["dist_km"] = pd.to_numeric(df["dist_km"], errors="coerce")
        df.loc[df["dist_km"] < 0, "dist_km"] = np.nan
    if "dur_min" in df.columns:
        df["dur_min"] = pd.to_numeric(df["dur_min"], errors="coerce")
        df.loc[df["dur_min"] < 0, "dur_min"] = np.nan

    return df


destinations_YS = build_dest_df(wege_short, "5-64", "w_rdist",     "dur_tmp")
destinations_OS = build_dest_df(wege_short, "65+",  "w_rdist",     "dur_tmp")
destinations_YL = build_dest_df(tages_long, "5-64", "dist_km_tmp", "dur_min_tmp")
destinations_OL = build_dest_df(tages_long, "65+",  "dist_km_tmp", "dur_min_tmp")

# =====================================================================
# 5) SAVE CSVs + SUMMARY
# =====================================================================
destinations_YS.to_csv("Trips/destinations_YS.csv", index=False)
destinations_OS.to_csv("Trips/destinations_OS.csv", index=False)
destinations_YL.to_csv("Trips/destinations_YL.csv", index=False)
destinations_OL.to_csv("Trips/destinations_OL.csv", index=False)

def quick_summary(name: str, df: pd.DataFrame):
    print(f"\n{name}: {len(df)} rows")
    if len(df) > 0:
        if "HHNR" in df.columns:
            n_persons = df[["HHNR", "survey"]].drop_duplicates().shape[0]
            print("unique persons (HHNR+survey):", n_persons)
        if "start_lang" in df.columns:
            print("start_lang missing %:", float(df["start_lang"].isna().mean() * 100))
        if "dest_lang" in df.columns:
            print("dest_lang  missing %:", float(df["dest_lang"].isna().mean() * 100))
        same = (df["orig_zone"] == df["dest_zone"]).mean() * 100
        print(f"orig==dest share %: {same:.2f}")
    else:
        print("(empty)")

for name, df in [
    ("destinations_YS", destinations_YS),
    ("destinations_OS", destinations_OS),
    ("destinations_YL", destinations_YL),
    ("destinations_OL", destinations_OL),
]:
    quick_summary(name, df)
